<a href="https://colab.research.google.com/github/Preenon-ngs/Preenon-ngs/blob/main/Copy_of_%F0%9F%A7%ACOP2_RNA_seq_data%2C_scanpy%2C_adata%2C_cell_cycle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
alexandervc_open_problems_single_cell_perturbations_data_path = kagglehub.dataset_download('alexandervc/open-problems-single-cell-perturbations-data')
alexandervc_op2_rna_seq_data_to_sparse_matrix_path = kagglehub.notebook_output_download('alexandervc/op2-rna-seq-data-to-sparse-matrix')

print('Data source import complete.')


# What is about ?

Load scRNA-seq data and first look. Use basic scanpy pipeline and brief look on cell (proliferation) cycle.

We load data prepared in the notebook: https://www.kaggle.com/alexandervc/op2-rna-seq-data-to-sparse-matrix
    
Make first look on them following scanpy tutorial: https://scanpy-tutorials.readthedocs.io/en/latest/pbmc3k.html
        
        
Also brief look on cell (prolifiration) cycle - almost no prolifirating cells is seen


Versions:

    10 save created adata: "adata_OPSCP_rawcounts_sparse_240090_21255.h5ad", future versions will just load it
    7 major revision - cell types, drugs, analysis added , analysis of found by AmbrosM drugs leads to interesting observations
    
    3,4,5,6 cosmetic changes
    2 use second version of prepared data - CSR matrix sparse format, and other small changes.
        Relax filtering by MT and n_genes_by_counts
    
    1 data from the first version of the notebook above, we need to update since we changed the format in version 2 of the data prepration nb.  
        default thresholds on MT and n_genes_by_counts are too restrictive - will change

# Preliminaries / Scanpy

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import time
t0start = time.time()

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install scanpy

In [ ]:
import scanpy as sc

# Load / Create adata

"adata" is "dataframe on steroids" - the standard format used in single-cell RNA seq data processing in Python.

The idea is that both "index" and "columns" of the "adata(steroid dataframe)" are pandas dataframes themselves - so allow to store lots of information.

While the "main-meat" is stored in adata.X - analogue of df.values, which can support "X" to be sparse matrix, pay attention it can only store numeric values in contrast to pandas dataframe (all non-numeric should be moved to "index = adata.obs").

Vocabluary:

    df.values <-> adata.X (can be sparse or numpy but only supports numeric values)
    df.index <-> adata.obs - it is a kind of "extended index" which is dataframe by itself
    df.columns <-> adata.var - it is a kind of "extened columns" which is dataframe by itself
    
    
See docs: https://scanpy.readthedocs.io/en/stable/usage-principles.html    

In [ ]:
%%time

if 1: # Load prepared adata
    fn = '/kaggle/input/open-problems-single-cell-perturbations-data/adata_OPSCP_rawcounts_sparse_240090_21255.h5ad'
    print(fn)
    adata = sc.read_h5ad(fn)
    print(adata)

if 0: # Here is code to prepare adata. We run it in versions 1-10, later saved it to Kaggle dataset and now use prepared file
    from scipy.sparse import load_npz

    fn = '/kaggle/input/op2-rna-seq-data-to-sparse-matrix/rna_seq_counts_sparse_matrix.npz'
    # Load the sparse matrix from the saved npz file
    X = load_npz(fn)
    print(type(X))
    # X = X.tocsr()
    print(X.shape)
    X[:3,:2].toarray()

    # %%time
    # Create an AnnData object with the sparse matrix
    adata = sc.AnnData(X=X)

    # %%time
    fn = '/kaggle/input/op2-rna-seq-data-to-sparse-matrix/rna_seq_obs_id.csv'
    df_obs = pd.read_csv(fn,index_col = 0)
    print(df_obs.shape)
    df_obs.columns = ['index cell']
    # df_obs = df_obs.set_index('cell_id')
    # df_obs['index'] = range(len(df_obs))
    #display(df_obs)

    fn = '/kaggle/input/open-problems-single-cell-perturbations/adata_obs_meta.csv'
    df_obs2 = pd.read_csv(fn, index_col = 0)
    print(df_obs2['cell_type'].unique() )
    #df_obs2
    df_obs = df_obs.join(df_obs2, how = 'left')
    display(df_obs)


    # %%time
    m = df_obs['sm_name'] == 'Dimethyl Sulfoxide'
    df_obs['control negative'] = (m).astype(float)
    print( df_obs['control negative'].sum() )

    m = df_obs['sm_name'] == 'Dabrafenib'
    m = m | ( df_obs['sm_name'] == 'Belinostat' )
    df_obs['control positive'] = (m).astype(float)
    print( df_obs['control positive'].sum() )


    # %%time
    fn = '/kaggle/input/op2-rna-seq-data-to-sparse-matrix/rna_seq_gene_symbols.csv'
    df_var = pd.read_csv(fn,index_col = 0)
    print(df_var.shape)
    df_var.columns = ['index gene']
    # df_var = df_var.set_index('cell_id')
    # df_var['index gene'] = range(len(df_var))
    df_var

    # %%time
    adata.obs = df_obs
    adata.var = df_var
    adata

    # %%time
    if 0:
        adata.write('adata_OPSCP_rawcounts_sparse_240090_21255.h5ad', compression='gzip')
    # %%time
    if 0:
        loaded_adata = sc.read_h5ad('adata_OPSCP_rawcounts_sparse_240090_21255.h5ad')
        print(loaded_adata)
        adata = loaded_adata


In [ ]:
adata.obs

In [ ]:
adata.var

In [ ]:
m = adata.obs['control'] == True
print(m.sum())
print( adata.obs[m]['sm_name'].value_counts() )

print( list(adata.obs['sm_name'].unique() ) )
adata.obs['sm_name'].value_counts()


## Some examples to operate with adata

In [ ]:
adata[:,'E2F1'].X.toarray().sum()

In [ ]:
adata[:,['E2F1','E2F2']].X.toarray().sum(axis = 0)

In [ ]:
adata[:100,:100].to_df()

# Genes not exrpessed in certain cell types

In [ ]:
%%time
list_cell_types = adata.obs['cell_type'].unique()
for cell_type in list_cell_types:
    m0 = adata.obs['cell_type'] == cell_type
    print(cell_type,'count cells:',  m0.sum() )
    v2 = (adata[m0,:].X!=0).sum(axis = 0 )# .
    k = 0
    m = np.asarray(v2).ravel() == k
    print(m.sum(), 'genes expressed in  exactly ', k, 'cells for cell type', cell_type  )
    print('50 genes:',  list(adata.var[m].index)[:50] )
    print()


# Least and top expressed genes

In [ ]:
%%time
v2 = (adata.X!=0).sum(axis = 0 )
v2 = np.asarray(v2).ravel()
print('Statistics: number of cells , where gene is non-zero - distribution over all genes')
print(pd.Series(v2).describe() )
plt.hist(v2 , bins = 100)
plt.title('number of cells , where gene is non-zero - distribution over all genes',fontsize = 17)
plt.show()
print('Top frequent - numbers - how many cells gene is expressed in:')
print( pd.Series(v2).value_counts().head(20) )
print(np.sort(v2)[:30])
print(np.sort(v2)[-30:])
print()


print('Least expressed genes (by number of cells)')
for k in [1,2,3,4,5,10,100]:
    m = np.asarray(v2).ravel() == k
    print(m.sum(), 'genes expressed in  exactly ', k, 'cells ')
    print('50 genes:',  list(adata.var[m].index)[:50] )
    print()

print(); print()
print('Top expressed genes (by number of cells)')
for k in [ 240_000, 230_000]:
    m = np.asarray(v2).ravel() >= k
    print(m.sum(), 'genes expressed in  >=  ', k, 'cells ')
    print('50 genes:',  list(adata.var[m].index)[:50] )
    print()


# Start analysis (standard pipeline)

Standard single cell rna seq data analysis piplene steps, see scanpy tutorial: https://scanpy-tutorials.readthedocs.io/en/latest/pbmc3k.html


In [ ]:
sc.pl.highest_expr_genes(adata, n_top=40, )

# Filtering

In [ ]:
%%time
print(adata.shape)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(adata.shape)


In [ ]:
%%time
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
%%time
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
%%time
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')


In [ ]:
%%time
# Changed to more relaxed , defuault values too restrictive
print(adata.shape)
adata = adata[adata.obs.n_genes_by_counts < 10_000, :]
adata = adata[adata.obs.pct_counts_mt < 20, :]
print(adata.shape)


# Norm & Log - preprocessing

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)

# PCA

In [ ]:
%%time
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)


In [ ]:
%%time
sc.pl.pca(adata, color=['CST3','MALAT1','B2M'] )
sc.pl.pca(adata, color=['cell_type', 'control negative', 'control positive' ])
sc.pl.pca(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])



# UMAP

In [ ]:
%%time
!pip install leidenalg

In [ ]:
import leidenalg

In [ ]:
%%time
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.leiden(adata)
sc.tl.paga(adata)
sc.pl.paga(adata, plot= False)  # remove `plot=False` if you want to see the coarse-grained graph
sc.tl.umap(adata, init_pos='paga')

In [ ]:
%%time
sc.tl.umap(adata)
sc.pl.umap(adata, color=['CST3', 'NKG7', 'PPBP'])

In [ ]:
adata

In [ ]:
%%time
if 0:
    adata.write('adata_OPSCP_FilteredNormalized_sparse_'+str(adata.shape[0])+'_'+str(adata.shape[1])+'.h5ad', compression='gzip')


In [ ]:
%%time
sc.pl.umap(adata, color=['CST3','MALAT1','B2M'] )
sc.pl.umap(adata, color=['cell_type', 'control negative', 'control positive' ])
sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])


# UMAP Cell types, etc

In [ ]:
print(adata.obs.columns)

In [ ]:
%%time

print( adata.obs['cell_type'].value_counts() )

adata.obs['cell_type cat']= adata.obs['cell_type'].astype('category')
adata.obs['cell_type T regulatory cells']= (adata.obs['cell_type'] == 'T regulatory cells').astype(float) # .astype('category')
print(adata.obs['cell_type T regulatory cells'].sum() )
adata.obs['cell_type T cells CD4+']= (adata.obs['cell_type'] == 'T cells CD4+').astype(float) # .astype('category')
print(adata.obs['cell_type T cells CD4+'].sum() )
adata.obs['cell_type T cells CD8+']= (adata.obs['cell_type'] == 'T cells CD8+').astype(float) # .astype('category')
print(adata.obs['cell_type T cells CD8+'].sum() )
adata.obs['cell_type NK cells']= (adata.obs['cell_type'] == 'NK cells').astype(float) # .astype('category')
print(adata.obs['cell_type NK cells'].sum() )
adata.obs['cell_type B cells']= (adata.obs['cell_type'] == 'B cells').astype(float) # .astype('category')
print(adata.obs['cell_type B cells'].sum() )
adata.obs['cell_type Myeloid cells']= (adata.obs['cell_type'] == 'Myeloid cells').astype(float) # .astype('category')
print(adata.obs['cell_type Myeloid cells'].sum() )

import warnings

warnings.simplefilter("ignore")

sc.pl.umap(adata, color=[ 'leiden', 'donor_id', 'cell_type'  ]) # n_genes', 'n_genes_by_counts', 'total_counts'])
sc.pl.umap(adata, color=['cell_type T regulatory cells','cell_type T cells CD4+','cell_type T cells CD8+']) # n_genes', 'n_genes_by_counts', 'total_counts'])
sc.pl.umap(adata, color=['cell_type NK cells','cell_type B cells','cell_type Myeloid cells']) # n_genes', 'n_genes_by_counts', 'total_counts'])

# sc.pl.umap(adata, color=['cell_type cat','total_counts_mt','sm_name']) # n_genes', 'n_genes_by_counts', 'total_counts'])
sc.pl.umap(adata, color=['n_genes', 'n_genes_by_counts', 'total_counts'])

sc.pl.umap(adata, color=['library_id', 'plate_name', 'well', 'row', 'col',]) # n_genes', 'n_genes_by_counts', 'total_counts'])



# AmbrosM found drugs (misclassified cell types) and other drugs - UMAP

    'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)'
    
indicated by AmbrosM as potentially leading to misclassified cell types - see
https://www.kaggle.com/competitions/open-problems-single-cell-perturbations/discussion/458661


In [ ]:
%%time
print(adata.obs['sm_name'].value_counts().head(10))
print(adata.obs['sm_name'].value_counts().tail(10))

print(adata.obs['sm_name'].value_counts().head(10).index.to_list() )

print(adata.obs['sm_name'].unique().to_list() )

list_selected_drugs = ['Dimethyl Sulfoxide', 'Dabrafenib', 'Belinostat', 'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)',]
for drug in  list_selected_drugs:
    adata.obs[drug] = (adata.obs['sm_name'] == drug).astype('category')
    print(drug, (adata.obs['sm_name'] == drug).sum() )

In [ ]:
import warnings

warnings.simplefilter("ignore")

In [ ]:
%%time
sc.pl.umap(adata, color=['donor_id', 'cell_type'  ]) # n_genes', 'n_genes_by_counts', 'total_counts'])

sc.pl.umap(adata, color=list_selected_drugs[:3]) # n_genes', 'n_genes_by_counts', 'total_counts'])
sc.pl.umap(adata, color=list_selected_drugs[3:7]) # n_genes', 'n_genes_by_counts', 'total_counts'])
sc.pl.umap(adata, color=list_selected_drugs[7:]) # n_genes', 'n_genes_by_counts', 'total_counts'])

sc.pl.umap(adata, color=['donor_id', 'cell_type','cell_type T cells CD8+' ,'cell_type T cells CD4+'  ]) # n_genes', 'n_genes_by_counts', 'total_counts'])


## Seaborn scatterplots

In [ ]:
%%time
x = adata.obsm['X_umap'][:,0]
y = adata.obsm['X_umap'][:,1]
sns.scatterplot(x=x,y=y, hue = adata.obs['cell_type'])
plt.show()
# %%time
list_selected_drugs = ['Dimethyl Sulfoxide', 'Dabrafenib', 'Belinostat', 'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)',]
for drug in  list_selected_drugs:
    adata.obs[drug] = (adata.obs['sm_name'] == drug).astype('category')
    print(drug, (adata.obs['sm_name'] == drug).sum() )

x = adata.obsm['X_umap'][:,0]
y = adata.obsm['X_umap'][:,1]
for col in list_selected_drugs:
    sns.scatterplot(x=x,y=y, hue = adata.obs[col])
    plt.show()

# Other drugs - most cannot see on umap - since not so many cells and effect is not so strong to produce separate cluster

In [ ]:
%%time
import warnings
warnings.simplefilter("ignore")

print(adata.obs['sm_name'].value_counts().head(10))
print(adata.obs['sm_name'].value_counts().tail(10))

#print(adata.obs['sm_name'].unique().to_list() )

list_selected_drugs = adata.obs['sm_name'].value_counts().index.to_list() # ['Dimethyl Sulfoxide', 'Dabrafenib', 'Belinostat', 'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)',]

print( list_selected_drugs )

for drug in  list_selected_drugs:
    adata.obs[drug] = (adata.obs['sm_name'] == drug).astype('category')
    print(drug, (adata.obs['sm_name'] == drug).sum() )

sc.pl.umap(adata, color= list_selected_drugs ) # n_genes', 'n_genes_by_counts', 'total_counts'])


## Seaborn scatterplots

In [ ]:
%%time
import warnings
warnings.simplefilter("ignore")

x = adata.obsm['X_umap'][:,0]
y = adata.obsm['X_umap'][:,1]
sns.scatterplot(x=x,y=y, hue = adata.obs['cell_type'])
plt.show()
# %%time
# list_selected_drugs = ['Dimethyl Sulfoxide', 'Dabrafenib', 'Belinostat', 'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)',]
list_selected_drugs = adata.obs['sm_name'].value_counts().head(12).index.to_list() # ['Dimethyl Sulfoxide', 'Dabrafenib', 'Belinostat', 'CGM-097', 'LY2090314','Ganetespib (STA-9090)','IN1451', 'Oprozomib (ONX 0912)','MLN 2238','CEP-18770 (Delanzomib)',]

x = adata.obsm['X_umap'][:,0]
y = adata.obsm['X_umap'][:,1]
for drug in list_selected_drugs:
    adata.obs[drug] = (adata.obs['sm_name'] == drug).astype('category')
    print(drug, (adata.obs['sm_name'] == drug).sum() )
    sns.scatterplot(x=x,y=y, hue = adata.obs[drug])
    plt.show()

# Some drug groups

In [ ]:
l = [t[-3:] for t in adata.obs['sm_name'].unique() ]
pd.Series(l).value_counts().head(10)

In [ ]:
[t for t in adata.obs['sm_name'].unique() if t[-3:] == 'one' ], [t for t in adata.obs['sm_name'].unique() if t[-3:] == 'ine' ],  [t for t in adata.obs['sm_name'].unique() if t[-3:] == 'tat'], [t for t in adata.obs['sm_name'].unique() if t[-3:] == 'sib'],

In [ ]:
%%time
dict_drug_groups = {}
dict_drug_groups['tyrosine kinase inhibitors'] = [t for t in adata.obs['sm_name'].unique() if t[-3:] == 'nib' ]
print(dict_drug_groups['tyrosine kinase inhibitors'])
# k = 'statins(Lower cholesterol in blood. Inhibit enzyme HMG-CoA reductase)' - mistake of chatGPT
k  = 'Belinostat Vorinostat Resminostat Ricolinostat Tosedostat'
dict_drug_groups[k] = [t for t in adata.obs['sm_name'].unique() if t[-4:] == 'stat' ]
print(dict_drug_groups[k])
k  = 'Vorinostat Resminostat Ricolinostat Tosedostat'
dict_drug_groups[k] = ['Vorinostat','Resminostat','Ricolinostat','Tosedostat' ]
print(dict_drug_groups[k])
k  = 'Idelalisib Dactolisib (PI3K inhibitors)'
dict_drug_groups[k] = [t for t in adata.obs['sm_name'].unique() if t[-3:] == 'sib' ]
print(dict_drug_groups[k])

# k = 'tyrosine kinase inhibitors'
for k in dict_drug_groups.keys():
    print(k)
    l = dict_drug_groups[k]
    print(l)
    v = adata.obs['sm_name'].isin(l).astype('category')
    adata.obs[k] = v
    sc.pl.umap(adata, color= k ) # n_genes', 'n_genes_by_counts', 'total_counts'])
    plt.show()



# Cell (prolifiration) Cycle G1S/G2M plot  - small number of proliferating cells

Proliferation cycle or cell cycle - key process for many cells. Many drugs here are anti-cancer - so should affect the ability of cells to proliferate. So might be interesting to look on drugs affect on these particular genes.

Tirosh et.al. proposed list of about 100 cell-cycle genes which are the most effectively seen by single cell data. It is good starting point.

In general there are much more genes related to the cell cycle, with various degree of "relation". Some list are cell cycle genes may contain thousands genes, but in fact in such huge lists most of the genes are related to cell cycle very weakly or these genes not good captured by single cell sequencing, while Tirosh list contains genes very strongly related to cell cycle and well captured by single cell technology. Moreover, many genes are associated to various biological processes in the cell, but Tirosh genes are mostly associated with cell cycle, not with the other processes - another argument why they are convenient to work with.

One may look at Computational challenges of cell cycle analysis using single cell transcriptomics Alexander Chervov, Andrei Zinovyev https://arxiv.org/abs/2208.05229

And hundreds Kaggle notebooks/datasets related to that work e.g. that discussion: https://www.kaggle.com/competitions/open-problems-multimodal/discussion/350314 , or that notebook: https://www.kaggle.com/code/alexandervc/mmscel-cell-cycle-03b-daybydaychange-allcelltypes/notebook

PS

Other cell cycle genes sets e.g. by Tom Freeman

See some comparison e.g. here: https://www.kaggle.com/code/alexandervc/tabmuris-cell-cycle-1-data-one-by-one#Cell-cycle So list is bigger but kind of more "dirty", that means containing more non-cell cycle effects, and G1S - G2M split is less prominent.

In [ ]:
G1S_genes_Tirosh = ['MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG', 'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP', 'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76', 'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51', 'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM', 'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8']
G2M_genes_Tirosh = ['HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A', 'NDC80', 'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF', 'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB', 'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP', 'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1', 'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR', 'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF', 'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA']
genes_Tirosh = G1S_genes_Tirosh + G2M_genes_Tirosh

# Subset of Tirosh genes to capture "fast" cell cycle pattern = see https://arxiv.org/abs/2208.05229
list_genes_fastCCsign = ['CDK1', 'UBE2C', 'TOP2A', 'TMPO', 'HJURP', 'RRM1', 'RAD51AP1', 'RRM2', 'CDC45', 'BLM', 'BRIP1', 'E2F8', 'HIST2H2AC']

G1S_genes_Freeman = ['ADAMTS1', 'ASF1B', 'ATAD2', 'BARD1', 'BLM', 'BRCA1', 'BRIP1', 'C17orf75', 'C9orf40', 'CACYBP', 'CASP8AP2', 'CCDC15', 'CCNE1', 'CCNE2', 'CCP110', 'CDC25A', 'CDC45', 'CDC6', 'CDC7', 'CDK2', 'CDT1', 'CENPJ', 'CENPQ', 'CENPU', 'CEP57', 'CHAF1A', 'CHAF1B', 'CHEK1', 'CLSPN', 'CREBZF', 'CRYL1', 'CSE1L', 'DCLRE1B', 'DCTPP1', 'DEK', 'DERA', 'DHFR', 'DNA2', 'DNAJC9', 'DNMT1', 'DONSON', 'DSCC1', 'DSN1', 'DTL', 'E2F8', 'EED', 'EFCAB11', 'ENDOD1', 'ETAA1', 'EXO1', 'EYA2', 'EZH2', 'FAM111A', 'FANCE', 'FANCG', 'FANCI', 'FANCL', 'FBXO5', 'FEN1', 'GGH', 'GINS1', 'GINS2', 'GINS3', 'GLMN', 'GMNN', 'GMPS', 'GPD2', 'HADH', 'HELLS', 'HSF2', 'ITGB3BP', 'KIAA0101', 'KNTC1', 'LIG1', 'MCM10', 'MCM2', 'MCM3', 'MCM4', 'MCM5', 'MCM6', 'MCM7', 'MCMBP', 'METTL9', 'MMD', 'MNS1', 'MPP1', 'MRE11A', 'MSH2', 'MSH6', 'MYO19', 'NASP', 'NPAT', 'NSMCE4A', 'ORC1', 'OSGEPL1', 'PAK1', 'PAQR4', 'PARP2', 'PASK', 'PAXIP1', 'PBX3', 'PCNA', 'PKMYT1', 'PMS1', 'POLA1', 'POLA2', 'POLD3', 'POLE2', 'PRIM1', 'PRPS2', 'PSMC3IP', 'RAB23', 'RAD51', 'RAD51AP1', 'RAD54L', 'RBBP8', 'RBL1', 'RDX', 'RFC2', 'RFC3', 'RFC4', 'RMI1', 'RNASEH2A', 'RPA1', 'RRM1', 'RRM2', 'SLBP', 'SLC25A40', 'SMC2', 'SMC3', 'SSX2IP', 'SUPT16H', 'TEX30', 'TFDP1', 'THAP10', 'THEM6', 'TIMELESS', 'TIPIN', 'TMEM106C', 'TMEM38B', 'TRIM45', 'TRIP13', 'TSPYL4', 'TTI1', 'TUBGCP5', 'TYMS', 'UBR7', 'UNG', 'USP1', 'WDHD1', 'WDR76', 'WRB', 'YEATS4', 'ZBTB14', 'ZWINT']
G2M_genes_Freeman = ['ADGRE5', 'ARHGAP11A', 'ARHGDIB', 'ARL6IP1', 'ASPM', 'AURKA', 'AURKB', 'BIRC5', 'BORA', 'BRD8', 'BUB1', 'BUB1B', 'BUB3', 'CCNA2', 'CCNB1', 'CCNB2', 'CCNF', 'CDC20', 'CDC25B', 'CDC25C', 'CDC27', 'CDCA3', 'CDCA8', 'CDK1', 'CDKN1B', 'CDKN3', 'CENPE', 'CENPF', 'CENPI', 'CENPN', 'CEP55', 'CEP70', 'CEP85', 'CKAP2', 'CKAP5', 'CKS1B', 'CKS2', 'CTCF', 'DBF4', 'DBF4B', 'DCAF7', 'DEPDC1', 'DLGAP5', 'ECT2', 'ERCC6L', 'ESPL1', 'FAM64A', 'FOXM1', 'FZD2', 'FZD7', 'FZR1', 'GPSM2', 'GTF2E1', 'GTSE1', 'H2AFX', 'HJURP', 'HMGB2', 'HMGB3', 'HMMR', 'HN1', 'INCENP', 'JADE2', 'KIF11', 'KIF14', 'KIF15', 'KIF18A', 'KIF18B', 'KIF20A', 'KIF20B', 'KIF22', 'KIF23', 'KIF2C', 'KIF4A', 'KIF5B', 'KIFC1', 'KPNA2', 'LBR', 'LMNB2', 'MAD2L1', 'MELK', 'MET', 'METTL4', 'MIS18BP1', 'MKI67', 'MPHOSPH9', 'MTMR6', 'NCAPD2', 'NCAPG', 'NCAPG2', 'NCAPH', 'NDC1', 'NDC80', 'NDE1', 'NEIL3', 'NEK2', 'NRF1', 'NUSAP1', 'OIP5', 'PAFAH2', 'PARPBP', 'PBK', 'PLEKHG3', 'PLK1', 'PLK4', 'PRC1', 'PRR11', 'PSRC1', 'PTTG1', 'PTTG3P', 'RACGAP1', 'RAD21', 'RASSF1', 'REEP4', 'SAP30', 'SHCBP1', 'SKA1', 'SLCO1B3', 'SOGA1', 'SPA17', 'SPAG5', 'SPC25', 'SPDL1', 'STIL', 'STK17B', 'TACC3', 'TAF5', 'TBC1D2', 'TBC1D31', 'TMPO', 'TOP2A', 'TPX2', 'TROAP', 'TTF2', 'TTK', 'TUBB4B', 'TUBD1', 'UBE2C', 'UBE2S', 'VANGL1', 'WEE1', 'WHSC1', 'XPO1', 'ZMYM1']


## G1S-G2M plot - Tirosh signatures - standard ones

In [ ]:
%%time
G1S = np.asarray(adata[:,list(set(G1S_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()
G2M = np.asarray(adata[:,list(set(G2M_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()

plt.figure(figsize = (15,6) )
sns.scatterplot(x= G1S, y=G2M, hue = adata.obs['cell_type'])#  , palette = 'rainbow' )
plt.title('G1S/G2M plot. Tirosh  signatures - standard ones.', fontsize= 20)
plt.xlabel('G1S', fontsize= 20)
plt.ylabel('G2M', fontsize= 20)
plt.grid()
plt.show()

c = np.round( np.corrcoef(G1S,G2M)[0,1] ,2 )
print('Correlation G1S-G2M', c )

for col in ['control', 'control negative', 'control positive', 'sm_name']:
    plt.figure(figsize = (15,6) )
    sns.scatterplot(x= G1S, y=G2M, hue = adata.obs[col])#  , palette = 'rainbow' )
    plt.title('G1S/G2M plot. Tirosh  signatures - standard ones.', fontsize= 20)
    plt.xlabel('G1S', fontsize= 20)
    plt.ylabel('G2M', fontsize= 20)
    plt.grid()
    plt.show()

# plt.figure(figsize = (15,6) )
# sns.scatterplot(x= G1S, y=G2M, hue = adata.obs['sm_name'])#  , palette = 'rainbow' )
# plt.title('G1S/G2M plot. Tirosh  signatures - standard ones.', fontsize= 20)
# plt.xlabel('G1S', fontsize= 20)
# plt.ylabel('G2M', fontsize= 20)
# plt.grid()
# plt.show()


## Freeman G1S-G2M signatures

Note: typically Freeman and Tirosh singatures give similar results,

But Tirosh is in a sense smaller, more restrictive and cleaner set of genes which capture mainly cell cycle effects.

Freeman is more wider set of genes capturing not only cell cycle effects (which is not good for the task), but for very noisy data we one may hope averaging more genes might help to reduce noise. Thus Tirosh is the first (and the standard) choice, especially for the high quality data, Freeman can be checked for very noisy data.

In general Freeman G1S and G2M are more correlated than Tirosh - which is not good for the G1S/G2M plot, but it is a price to pay including more genes which might be helpful for noisy data.

In [ ]:
%%time
G1S = np.asarray(adata[:,list(set(G1S_genes_Freeman)&set(adata.var.index))].X.mean(axis = 1)).ravel()
G2M = np.asarray(adata[:,list(set(G2M_genes_Freeman)&set(adata.var.index))].X.mean(axis = 1)).ravel()
print( G1S.shape, G2M.shape)

plt.figure(figsize = (15,6) )
sns.scatterplot(x= G1S, y=G2M)
plt.title('G1S/G2M plot. Freeman signatures', fontsize= 20)
plt.xlabel('G1S', fontsize= 20)
plt.ylabel('G2M', fontsize= 20)
plt.grid()
plt.show()

c = np.round( np.corrcoef(G1S,G2M)[0,1] ,2 )
print('Correlation G1S-G2M', c )

## Fast signature plot

Useful to visualize "fast" cell cycle (it is not standard pattern of the cell cycle appearing for embryonic stem cells and some cancer cells (in particular many tp53-mutated ) ).  See https://arxiv.org/abs/2208.05229

In [ ]:
# Subset of Tirosh genes to capture "fast" cell cycle pattern = see https://arxiv.org/abs/2208.05229
list_genes_fastCCsign = ['CDK1', 'UBE2C', 'TOP2A', 'TMPO', 'HJURP', 'RRM1', 'RAD51AP1', 'RRM2', 'CDC45', 'BLM', 'BRIP1', 'E2F8', 'HIST2H2AC']

# G1S = np.asarray(adata[:,list(set(G1S_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()
G2M = np.asarray(adata[:,list(set(G2M_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()
fastCC = np.asarray(adata[:,list(set(list_genes_fastCCsign)&set(adata.var.index))].X.mean(axis = 1)).ravel()
print( G1S.shape, fastCC.shape)

plt.figure(figsize = (15,6) )
sns.scatterplot(x= fastCC, y=G2M, hue = adata.obs['cell_type'])
plt.title('fastCC-G2M signatures', fontsize= 20)
plt.xlabel('fastCC', fontsize= 20)
plt.ylabel('G2M', fontsize= 20)
plt.grid()
plt.show()

c = np.round( np.corrcoef(G1S,fastCC)[0,1] ,2 )
print('Correlation G1S-fastCC', c )

## UMAP Colored by cell cycle signatures

Very few cells are proliferating -  the cluster is very small.


In [ ]:
G1S = np.asarray(adata[:,list(set(G1S_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()
G2M = np.asarray(adata[:,list(set(G2M_genes_Tirosh)&set(adata.var.index))].X.mean(axis = 1)).ravel()
adata.obs['G1S'] = G1S
adata.obs['G2M'] = G2M
adata.obs['G1S_bin'] = (G1S > 0.4)
adata.obs['G1S_bin'] = adata.obs['G1S_bin'].apply(lambda x: str(x)).astype('category')


sc.pl.umap(adata, color=['G1S_bin', 'G1S', 'G2M',])

In [ ]:
%%time
x = adata.obsm['X_umap'][:,0]
y = adata.obsm['X_umap'][:,1]
sns.scatterplot(x=x,y=y, hue = adata.obs['G1S_bin'])
plt.show()

In [ ]:
print('%.1f seconds passed total '%(time.time()-t0start) )
print('%.1f minutes passed total '%( (time.time()-t0start)/60)  )
print('%.2f hours passed total '%( (time.time()-t0start)/3600)  )